# Prompt Evaluations - Grader Pipeline

This notebook implements an evaluation pipeline for AWS tasks: it generates (or loads) test cases, runs model responses, and assigns automatic scores for each output.

## Environment Setup

Loads environment variables, initializes the Anthropic client, and defines the model used in all requests.

In [1]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

## Helper Functions

Centralizes utilities to build message history and send model requests with control over system, temperature, and stop sequences.

In [2]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

## Generate Evaluation Dataset

Defines a prompt to create short AWS task examples (Python, JSON, or Regex), ensuring JSON output ready for evaluation.

In [3]:
# Function to generate a new dataset
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
        "format": "json" or "python" or "regex"
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

## Persist Dataset

Runs test case generation and saves results to ../data/dataset.json for reuse in future runs.

In [4]:
# Generate the dataset and write it to '../data/dataset.json'
dataset = generate_dataset()
with open("../data/dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

## Grade Output with Model

Defines the automatic evaluator that compares task and solution, returning strengths, weaknesses, reasoning, and a numeric score in JSON.

In [5]:
# Function to grade a test case + output using a model
def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

## Run Prompt per Test Case

Wraps execution of the main prompt for each dataset item and returns the model raw response for evaluation.

In [6]:
# Passes a test case into Claude
def run_prompt(test_case):
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""

    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

## Alternate Grader Variant

Alternative grading function version with shorter review instructions. Useful for testing differences in criteria between evaluation prompts.

In [16]:
def grade_by_model(test_case, output):
    # Create evaluation prompt
    eval_prompt = f"""
    You are an expert code reviewer. Evaluate this AI-generated solution.
    
    Task: {test_case["task"]}
    Solution: {output}
    
    Provide your evaluation as a structured JSON object with:
    - "strengths": An array of 1-3 key strengths
    - "weaknesses": An array of 1-3 key areas for improvement  
    - "reasoning": A concise explanation of your assessment
    - "score": A number between 1-10
    """
    
    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)


## Run a Single Evaluation

Runs a single test case end-to-end: generates output, applies grading, and returns a consolidated structure with score and reasoning.

In [12]:
# Function to execute a single test case and grade the output
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    model_grade = grade_by_model(test_case, output)
    score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning,
    }

## Run Full Evaluation

Iterates through the entire dataset, runs each test case, and calculates the overall average score to measure prompt quality in aggregate.

In [20]:
from statistics import mean


def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")

    return results

## Load Dataset and Execute

Loads the dataset saved on disk and triggers full evaluation execution to generate the results list.

In [21]:
with open("../data/dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

Average score: 7.333333333333333


## Pretty Print Results

Displays results as formatted JSON to make reading, case comparison, and pipeline debugging easier.

In [22]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# AWS S3 Bucket Name Parser\n\nHere are several ways to extract the bucket name from an S3 URI:\n\n## Method 1: Simple String Manipulation (Recommended)\n```python\ndef get_bucket_name(s3_uri):\n    \"\"\"Extract bucket name from S3 URI\"\"\"\n    # Remove 's3://' prefix and split by '/'\n    return s3_uri.replace('s3://', '').split('/')[0]\n\n# Test\nprint(get_bucket_name('s3://my-bucket/path/to/file.txt'))  # Output: my-bucket\nprint(get_bucket_name('s3://another-bucket/file.txt'))     # Output: another-bucket\n```\n\n## Method 2: Using Regular Expressions\n```python\nimport re\n\ndef get_bucket_name(s3_uri):\n    \"\"\"Extract bucket name using regex\"\"\"\n    match = re.match(r's3://([^/]+)', s3_uri)\n    return match.group(1) if match else None\n\n# Test\nprint(get_bucket_name('s3://my-bucket/path/to/file.txt'))  # Output: my-bucket\n```\n\n## Method 3: Using urllib (Most Robust)\n```python\nfrom urllib.parse import urlparse\n\ndef get_bucket_name(s3_uri):\n 